# AI vs Human Text Detection
### Ammar Yousuf Abrahani — Independent NLP Research Project

This notebook trains and evaluates multiple classifiers to distinguish **human-written** from **AI-generated** text.  
The focus is not just on accuracy — but on **interpretability**: understanding *why* the model makes the decisions it does.

---
**Research Questions:**
1. Can classical ML classifiers reliably detect AI-generated text?
2. Which linguistic features are most predictive?
3. Where does the model fail — and what does that tell us?


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from scipy.sparse import hstack, csr_matrix

print("✅ All libraries imported successfully")


## 2. Load Dataset

In [ ]:
df = pd.read_csv("data/dataset.csv")

print(f"Total samples : {len(df)}")
print(f"Human         : {(df.label==0).sum()}")
print(f"AI-Generated  : {(df.label==1).sum()}")
print()
df.head(6)


## 3. Exploratory Data Analysis

Let's look at the distribution and some example texts before training anything.


In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Human', 'AI-Generated'], 
            [(df.label==0).sum(), (df.label==1).sum()],
            color=['#4CAF50', '#F44336'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')

# Text length distribution
df['text_length'] = df['text'].apply(len)
axes[1].hist(df[df.label==0]['text_length'], bins=15, alpha=0.6, 
             label='Human', color='#4CAF50')
axes[1].hist(df[df.label==1]['text_length'], bins=15, alpha=0.6, 
             label='AI-Generated', color='#F44336')
axes[1].set_title('Text Length Distribution')
axes[1].set_xlabel('Characters')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nAvg human text length   : {df[df.label==0]['text_length'].mean():.0f} chars")
print(f"Avg AI text length      : {df[df.label==1]['text_length'].mean():.0f} chars")


In [ ]:
# Show sample texts
print("=== HUMAN TEXT EXAMPLE ===")
print(df[df.label==0]['text'].iloc[0])
print()
print("=== AI-GENERATED TEXT EXAMPLE ===")
print(df[df.label==1]['text'].iloc[0])


## 4. Linguistic Feature Engineering

Instead of feeding raw text directly, we extract **hand-crafted features** that capture the stylistic differences between human and AI writing.

This is the interpretable part — each feature has a clear linguistic meaning.


In [ ]:
def extract_features(texts):
    """
    Extract linguistic and stylistic features from text.
    Each feature is designed to capture a known difference
    between human and AI writing styles.
    """
    rows = []
    for text in texts:
        words     = text.split()
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]

        rows.append({
            # Word-level features
            'avg_word_len'      : np.mean([len(w) for w in words]) if words else 0,
            'vocab_richness'    : len(set(words)) / len(words) if words else 0,
            
            # Sentence-level features
            'avg_sent_len'      : np.mean([len(s.split()) for s in sentences]) if sentences else 0,
            'num_sentences'     : len(sentences),
            
            # Punctuation & style
            'punct_density'     : sum(1 for c in text if c in '.,;:!?') / len(text) if text else 0,
            'uppercase_ratio'   : sum(1 for c in text if c.isupper()) / len(text) if text else 0,
            'exclamation'       : text.count('!'),
            'question_marks'    : text.count('?'),
            'parenthetical'     : text.count('('),
            'ellipsis'          : text.count('...'),
            'em_dash'           : text.count('—') + text.count('--'),
            
            # Human markers (negative predictors of AI)
            'contractions'      : len(re.findall(r"\b\w+'\w+\b", text)),
            'filler_words'      : len(re.findall(r'\b(honestly|basically|actually|literally|just|like|tbh|lol|okay|ok)\b', text, re.I)),
            
            # AI markers (positive predictors of AI)
            'formal_connectors' : len(re.findall(r'\b(furthermore|moreover|however|therefore|consequently|nevertheless|in conclusion|in summary)\b', text, re.I)),
            'hedge_words'       : len(re.findall(r'\b(perhaps|may|might|could|suggest|indicate|appear|seem|likely)\b', text, re.I)),
            'has_numbering'     : int(bool(re.search(r'\b(first|second|third|1\.|2\.|3\.)', text, re.I))),
            'bullet_like'       : int(bool(re.search(r'^\s*[\-\*\•]|\b\d+[\.\_]\s', text, re.M))),
            'starts_with_i'     : int(bool(re.match(r'^I\b', text.strip()))),
            'avg_word_len_norm' : np.mean([len(w) for w in words]) / 10 if words else 0,
        })

    return pd.DataFrame(rows)

# Extract features
all_features = extract_features(df['text'])
all_features['label'] = df['label'].values

print("Features extracted:")
print(all_features.describe().round(3))


### 4.1 Visualise Key Feature Differences

Let's see how the features differ between human and AI text.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Linguistic Feature Distributions: Human vs AI', fontsize=13, fontweight='bold')

features_to_plot = [
    ('contractions',       'Contractions (it\'s, I\'ve)'),
    ('filler_words',       'Filler Words (honestly, tbh, lol)'),
    ('formal_connectors',  'Formal Connectors (furthermore, therefore)'),
    ('avg_word_len',       'Average Word Length'),
    ('avg_sent_len',       'Average Sentence Length'),
    ('hedge_words',        'Hedge Words (may, might, suggest)'),
]

human_feat = all_features[all_features.label == 0]
ai_feat    = all_features[all_features.label == 1]

for ax, (feat, title) in zip(axes.flat, features_to_plot):
    ax.hist(human_feat[feat], bins=10, alpha=0.6, label='Human',        color='#4CAF50')
    ax.hist(ai_feat[feat],    bins=10, alpha=0.6, label='AI-Generated', color='#F44336')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Value')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## 5. Train / Test Split

In [ ]:
X_text = df['text']
y      = df['label']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Training set : {len(X_train_text)} samples")
print(f"Test set     : {len(X_test_text)} samples")
print(f"\nTrain — Human: {(y_train==0).sum()} | AI: {(y_train==1).sum()}")
print(f"Test  — Human: {(y_test==0).sum()}  | AI: {(y_test==1).sum()}")


## 6. Build Feature Matrices

We use two types of features:
- **TF-IDF** — captures word/phrase patterns
- **Linguistic features** — our hand-crafted stylistic features
- **Combined** — both together


In [ ]:
# TF-IDF (unigrams + bigrams)
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=3000, sublinear_tf=True)
tfidf_train = tfidf.fit_transform(X_train_text)
tfidf_test  = tfidf.transform(X_test_text)

# Linguistic features
feat_train = extract_features(X_train_text.reset_index(drop=True))
feat_test  = extract_features(X_test_text.reset_index(drop=True))

# Combined matrix
X_train_combined = hstack([tfidf_train, csr_matrix(feat_train.values)])
X_test_combined  = hstack([tfidf_test,  csr_matrix(feat_test.values)])

print(f"TF-IDF matrix shape      : {tfidf_train.shape}")
print(f"Linguistic features shape : {feat_train.shape}")
print(f"Combined matrix shape     : {X_train_combined.shape}")


## 7. Train and Compare Models

In [ ]:
models = {
    "Logistic Regression\n(TF-IDF)":    (LogisticRegression(max_iter=1000), tfidf_train, tfidf_test),
    "Logistic Regression\n(Combined)":  (LogisticRegression(max_iter=1000), X_train_combined, X_test_combined),
    "Random Forest\n(Combined)":        (RandomForestClassifier(n_estimators=100, random_state=42), X_train_combined, X_test_combined),
    "Gradient Boosting\n(Combined)":    (GradientBoostingClassifier(n_estimators=100, random_state=42), X_train_combined, X_test_combined),
    "LinearSVC\n(TF-IDF)":              (LinearSVC(max_iter=2000), tfidf_train, tfidf_test),
}

results = {}
print(f"{'Model':<40} | {'Macro F1':>8} | {'Accuracy':>8}")
print("-" * 62)

for name, (model, X_tr, X_te) in models.items():
    model.fit(X_tr, y_train)
    preds = model.predict(X_te)
    f1    = f1_score(y_test, preds, average='macro')
    acc   = accuracy_score(y_test, preds)
    results[name] = {"model": model, "preds": preds, "f1": f1, "acc": acc, "X_te": X_te}
    print(f"{name.replace(chr(10), ' '):<40} | {f1:>8.3f} | {acc:>8.3f}")


## 8. Best Model — Detailed Results

In [ ]:
best_name = max(results, key=lambda k: results[k]['f1'])
best      = results[best_name]

print(f"Best model: {best_name.replace(chr(10), ' ')}")
print()
print(classification_report(y_test, best['preds'], 
                             target_names=['Human', 'AI-Generated']))


### 8.1 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Model comparison bar chart
names = [n.replace('\n', ' ') for n in results.keys()]
f1s   = [results[n]['f1'] for n in results.keys()]
colors = ['#2196F3' if f == max(f1s) else '#90CAF9' for f in f1s]
bars = axes[0].barh(names, f1s, color=colors)
axes[0].set_xlabel('Macro F1-Score')
axes[0].set_title('Model Comparison')
axes[0].set_xlim(0, 1.1)
for bar, val in zip(bars, f1s):
    axes[0].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)

# Confusion matrix
cm = confusion_matrix(y_test, best['preds'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Human', 'AI'], yticklabels=['Human', 'AI'])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()


## 9. Feature Importance — The Interpretability Part

This is the most important section for understanding *why* the model works.

We train a Logistic Regression on linguistic features only, so each coefficient 
directly tells us: **which features push toward AI vs Human prediction?**


In [ ]:
lr_feat = LogisticRegression(max_iter=1000)
lr_feat.fit(feat_train, y_train)

feat_importance = pd.DataFrame({
    'feature':     feat_train.columns.tolist(),
    'coefficient': lr_feat.coef_[0]
}).sort_values('coefficient', ascending=False)

print("Feature Importance")
print("Positive coefficient → predicts AI-Generated")
print("Negative coefficient → predicts Human")
print()
print(feat_importance.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#F44336' if c > 0 else '#4CAF50' for c in feat_importance['coefficient']]
bars = ax.barh(feat_importance['feature'], feat_importance['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Coefficient (positive = AI signal, negative = Human signal)')
ax.set_title('Linguistic Feature Importance\nWhat actually distinguishes AI from Human writing?', fontweight='bold')

# Add value labels
for bar, val in zip(bars, feat_importance['coefficient']):
    x = val + 0.02 if val >= 0 else val - 0.02
    ha = 'left' if val >= 0 else 'right'
    ax.text(x, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', ha=ha, fontsize=8)

plt.tight_layout()
plt.show()


## 10. Error Analysis

Where did the model fail? This is more interesting than the accuracy number.

Even if we have zero errors here — understanding *why* the dataset was easy is the real insight.


In [ ]:
test_df = X_test_text.reset_index(drop=True).to_frame()
test_df['true_label'] = y_test.reset_index(drop=True)
test_df['pred_label'] = best['preds']
test_df['correct']    = test_df['true_label'] == test_df['pred_label']

errors = test_df[~test_df['correct']]

print(f"Total test samples : {len(test_df)}")
print(f"Correct            : {test_df['correct'].sum()}")
print(f"Errors             : {len(errors)}")

if len(errors) == 0:
    print()
    print("✅ No errors on test set.")
    print()
    print("⚠️  BUT — this is worth thinking about critically:")
    print("   The model achieved perfect accuracy because human and AI samples")
    print("   had very distinct stylistic profiles in this dataset.")
    print()
    print("   In the real world, if an LLM is prompted to write casually —")
    print("   using contractions, filler words, typos — this classifier would likely fail.")
    print()
    print("   This is the deeper research challenge: surface-level detection is not enough.")
    print("   We need to understand what's happening INSIDE the model.")
else:
    for i, row in errors.iterrows():
        true = 'Human' if row['true_label'] == 0 else 'AI-Generated'
        pred = 'Human' if row['pred_label'] == 0 else 'AI-Generated'
        print(f"\n[Error #{i}] True: {true} → Predicted: {pred}")
        print(f"Text: {row['text'][:300]}...")


## 11. Key Takeaways

### What predicts AI-generated text
| Feature | Insight |
|---|---|
| `avg_word_len` | AI uses longer, more formal vocabulary |
| `num_sentences` | AI tends to write more structured, multi-sentence responses |
| `has_numbering` | AI loves numbered lists and structured formatting |
| `formal_connectors` | AI overuses *furthermore*, *consequently*, *in conclusion* |

### What predicts human text
| Feature | Insight |
|---|---|
| `contractions` | Humans write *it's*, *I've*, *can't* — AI writes *it is*, *I have* |
| `filler_words` | Humans write *honestly*, *basically*, *tbh*, *lol* |
| `em_dash` | Humans use em-dashes more naturally |
| `hedge_words` | Humans hedge differently and less formally |

---

### The Deeper Question
Perfect detection on this dataset doesn't mean the problem is solved.

As LLMs improve at mimicking human writing styles, surface-level features will become less reliable. The real research frontier is **mechanistic interpretability** — understanding what internal representations inside the model lead to these output patterns.

This connects directly to the PhD research area: using neural network interpretability and explainable AI techniques to understand LLM behaviour from the inside out.


## 12. Save Results

In [ ]:
import os
os.makedirs('outputs', exist_ok=True)

# Save feature importance
feat_importance.to_csv('outputs/feature_importance.csv', index=False)

# Save summary
summary = pd.DataFrame([{
    'best_model' : best_name.replace('\n', ' '),
    'macro_f1'   : round(best['f1'], 4),
    'accuracy'   : round(best['acc'], 4),
    'num_errors' : len(errors),
    'total_test' : len(test_df),
}])
summary.to_csv('outputs/summary.csv', index=False)

print("✅ Saved:")
print("   outputs/feature_importance.csv")
print("   outputs/summary.csv")
